In [63]:
import pyarrow.parquet as pq

table = pq.read_table("train-00000-of-00001.parquet")
df = table.to_pandas()




In [64]:
table.schema

text: string
label: int64
-- schema metadata --
huggingface: '{"info": {"features": {"text": {"dtype": "string", "_type":' + 59

In [65]:
df.isnull().sum()


text     0
label    0
dtype: int64

In [66]:
import pandas as pd
tf = pd.read_csv("cyberbullying_tanglish_binary_60k.csv")
tf.drop("id",axis=1,inplace=True)

In [67]:
df = df.join(tf.drop(columns=["text", "label"]))

In [68]:
tf.columns

Index(['text', 'label'], dtype='object')

In [69]:
import os
import nltk

local_nltk_path = os.path.join(os.getcwd(), "nltk_data")
os.makedirs(local_nltk_path, exist_ok=True)

nltk.data.path.append(local_nltk_path)

nltk.download("punkt", download_dir=local_nltk_path)
nltk.download("stopwords", download_dir=local_nltk_path)
nltk.download("punkt_tab", download_dir=local_nltk_path)
nltk.download("wordnet", download_dir=local_nltk_path)


[nltk_data] Downloading package punkt to
[nltk_data]     /Volumes/CrucialX9/Project/Data
[nltk_data]     Science/NLP/Cyberbulling detection /nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Volumes/CrucialX9/Project/Data
[nltk_data]     Science/NLP/Cyberbulling detection /nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Volumes/CrucialX9/Project/Data
[nltk_data]     Science/NLP/Cyberbulling detection /nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Volumes/CrucialX9/Project/Data
[nltk_data]     Science/NLP/Cyberbulling detection /nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [70]:


import re

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # remove URLs & mentions only
    text = re.sub(r"http\S+|www\S+|@\w+", "", text)

    # keep words and spaces
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [71]:
df["clean_text"] = df["text"].apply(preprocess_text)


In [72]:
X = df["clean_text"]
y = df["label"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 6),
    max_features=200_000,
    min_df=2
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


In [74]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train_vec, y_train)


/Volumes/CrucialX9/Project/Data Science/NLP/Cyberbulling detection /.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Volumes/CrucialX9/Project/Data Science/NLP/Cyberbulling detection /.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Volumes/CrucialX9/Project/Data Science/NLP/Cyberbulling detection /.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [75]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.8862503664614483

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.90      0.89     13644
           1       0.90      0.87      0.88     13644

    accuracy                           0.89     27288
   macro avg       0.89      0.89      0.89     27288
weighted avg       0.89      0.89      0.89     27288


Confusion Matrix:
 [[12284  1360]
 [ 1744 11900]]


In [76]:
from sklearn.svm import LinearSVC

sv = LinearSVC(class_weight="balanced")
sv.fit(X_train_vec, y_train)


,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,verbose,0
,random_state,None


In [77]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = sv.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.886946643213134

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.90      0.89     13644
           1       0.90      0.87      0.89     13644

    accuracy                           0.89     27288
   macro avg       0.89      0.89      0.89     27288
weighted avg       0.89      0.89      0.89     27288


Confusion Matrix:
 [[12276  1368]
 [ 1717 11927]]


In [78]:
samples = [
    "you are an idiot",
    "have a nice day everyone",
    "go back to where you came from",
    "i will kill you",
    "go to hell",
]

samples_clean = [preprocess_text(s) for s in samples]
samples_vec = vectorizer.transform(samples_clean)

print(model.predict(samples_vec))


[1 1 0 1 1]


In [79]:
import joblib

joblib.dump(model, "cyberbullying_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")


['tfidf_vectorizer.pkl']

In [80]:
import joblib

model = joblib.load("cyberbullying_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")


In [81]:
def predict_cyberbullying(texts):
    texts_clean = [preprocess_text(t) for t in texts]
    texts_vec = vectorizer.transform(texts_clean)
    return model.predict(texts_vec)


In [82]:
samples = [
    "you are so stupid",
    "thanks for your help today",
    "nobody likes you",
    "good morning everyone",
    "you are useless"
]

print(predict_cyberbullying(samples))


[1 0 1 0 1]


In [83]:
You’re useless, nobody wants you here.
I disagree with your opinion
Nice post, thanks for sharing.

SyntaxError: invalid character '’' (U+2019) (983374245.py, line 1)